# Lección 8: Análisis Numérico y de los Errores
### Cálculo I y Fundamentos de la Física Matemática

---

Este cuaderno de Jupyter establece la **fundamentación formal del análisis numérico y la teoría de errores**. Abordamos el origen histórico y la evolución de la disciplina, el diseño y condicionamiento de algoritmos, y la aritmética de punto flotante de las computadoras. Estudiamos detalladamente la clasificación y cuantificación de errores (de modelo, truncamiento y redondeo) y las consecuencias del orden de las operaciones en la propagación de errores. Asimismo, analizamos la convergencia, estabilidad y el criterio de Scarborough. Concluimos con el análisis formal de los polinomios de Taylor y Maclaurin, demostrando analíticamente el residuo integral e ilustrando la estimación del error mediante la cota de Lagrange para funciones trigonométricas.

---


## Objetivos de Aprendizaje

Al finalizar esta lección, serás capaz de:
1. **Comprender** la necesidad de aplicar métodos numéricos en ingeniería para resolver ecuaciones polinómicas complejas y ecuaciones trascendentes.
2. **Identificar** la evolución del análisis numérico a lo largo de la historia y su paralelismo con el desarrollo de la computación.
3. **Analizar** el error de cancelación o pérdida de relevancia al realizar operaciones aritméticas entre números de magnitudes extremas o muy cercanas.
4. **Diferenciar y cuantificar** los errores absolutos, relativos y porcentuales, aplicando las reglas de redondeo simétrico al par más cercano.
5. **Evaluar** la propagación del error y el impacto del orden de las operaciones (suma ascendente vs. descendente) en la precisión del resultado.
6. **Demostrar** analíticamente la forma integral del residuo de Taylor y estimar cotas de error usando el teorema de Lagrange.


## 1. Introducción e Historia del Análisis Numérico

### 1.1 ¿Qué es el Análisis Numérico?
El **análisis numérico** es la rama de las matemáticas que se encarga de describir, analizar y diseñar **algoritmos numéricos** para resolver problemas matemáticos continuos (como raíces de ecuaciones, sistemas lineales, integración y ecuaciones diferenciales) en los que están involucradas cantidades numéricas con una precisión determinada. 

A diferencia del análisis analítico que busca soluciones exactas (fórmulas cerradas), los métodos numéricos son **procesos iterativos** que buscan aproximaciones numéricas a soluciones específicas, controlando el error a través de la repetición de un proceso.

### 1.2 Necesidad en la Ingeniería y Ciencias
En cursos de álgebra elemental se resuelven polinomios de grado $n$ mediante división sintética o fórmulas cerradas. Sin embargo, para polinomios de grado alto ($n \ge 5$) no existen fórmulas generales (Teorema de Abel-Ruffini). Además, para **ecuaciones trascendentes** (como $x e^x - 1 = 0$ o $\cos(x) - x = 0$) es imposible hallar una solución analítica exacta. Los métodos numéricos proporcionan aproximaciones de alta precisión esenciales para simular trayectorias espaciales, calcular precios de derivados financieros, optimizar rutas aéreas o diseñar vehículos mediante simulaciones estructurales.

### 1.3 Cronología y Evolución Histórica
La historia del análisis numérico corre en paralelo al desarrollo de herramientas de cálculo:
- **1650 a. C. (Papiro de Rhind)**: Primeros métodos aritméticos para resolver expresiones matemáticas sin álgebra.
- **250 a. C. (Euclides y Arquímedes)**: Uso del **método de exhaución** para acotar geométricamente el valor de $\pi$ mediante polígonos regulares inscritos y circunscritos.
- **Siglo IX (Al-Juarismi)**: Formalización de los **algoritmos** (reglas de cálculo ordenadas y finitas).
- **1768 (Leonhard Euler)**: Introducción de la integración numérica y aproximación de ecuaciones diferenciales ordinarias (Método de Euler). Jacob Stirling y Brook Taylor desarrollan la teoría de diferencias finitas.
- **1822-1843 (Babbage y Lovelace)**: Concepción de la máquina diferencial y la máquina analítica, publicando las primeras notas sobre programación computacional.
- **1937-1945 (Turing y Von Neumann)**: Conceptualización del computador universal, descifrado de códigos de guerra en Bletchley Park y uso de computadoras analógicas y electrónicas (EDVAC) para cálculos balísticos.
- **1950-1953 (Wilkinson y Backus)**: Desarrollo de subrutinas de álgebra matricial (ACE) y creación de **FORTRAN**, primer lenguaje compilado de alto nivel.
- **1970-1984 (EISPACK, LINPACK y MATLAB)**: Creación de bibliotecas estandarizadas en Fortran para cálculo numérico y posterior lanzamiento de MATLAB por Cleve Moler como un entorno integrado para álgebra lineal sin necesidad de programar en Fortran.


## 2. Aritmética de Punto Flotante y Pérdida de Significancia

### 2.1 Épsilon de Máquina e Indeterminación
Las computadoras representan los números reales en formato de **punto flotante** mediante una mantisa y un exponente (estándar IEEE 754). La precisión está limitada por el **épsilon de máquina** ($\epsilon_m \approx 2.22 \times 10^{-16}$ en precisión doble), que es el número positivo más pequeño tal que la máquina distingue entre $1$ y $1 + \epsilon_m$.

### 2.2 Error de Cancelación
El **error de cancelación** (o pérdida de relevancia/significancia) ocurre cuando se restan dos números de magnitud similar o cuando se suma un número muy pequeño a uno muy grande. Al hacer esto, los dígitos significativos de menor peso se desplazan fuera de la mantisa representable y se pierden.

**Ejemplo analítico**: Evaluar el comportamiento de la función:
$$f(x) = \frac{(1+x)-1}{x}$$
cuando $x$ tiende a cero. Analíticamente, $f(x) = 1$ para todo $x \neq 0$. Sin embargo, en aritmética de punto flotante de doble precisión:
- Si $x = 10^{-15}$, la suma $1 + x$ se representa internamente truncada debido al espacio finito de la mantisa.
- Al restar $1$ a $(1+x)$, se eliminan los dígitos principales de coincidencia, dejando una mantisa ruidosa y con muy pocos dígitos significativos.
- Al dividir finalmente por $x$, obtenemos un valor que difiere de 1, revelando la precisión de la máquina.

### 2.3 Cancelación Restativa en la Serie Exponencial
Consideremos el cálculo de $e^{-5.5}$ utilizando su desarrollo en serie de Taylor:
$$e^{-5.5} = 1 - 5.5 + \frac{5.5^2}{2!} - \frac{5.5^3}{3!} + \dots$$
El valor real es $e^{-5.5} \approx 0.00408677$. Si calculamos la suma directamente sumando y restando términos:
- Los términos intermedios son muy grandes (ej. $\frac{5.5^5}{5!} \approx 41.942$).
- Sumar y restar números del orden de $40$ para obtener un resultado final del orden de $0.004$ provoca que los errores de redondeo acumulados en los dígitos de menor peso destruyan la precisión del resultado final.
- **Solución numérica**: Utilizar la identidad de reciprocidad:
  $$e^{-5.5} = \frac{1}{e^{5.5}} = \frac{1}{\sum_{k=0}^\infty \frac{5.5^k}{k!}}$$
  En este caso, todos los términos sumados son positivos, por lo que no hay cancelaciones restativas y el resultado es numéricamente estable.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Configuración de estilo visual premium
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})

# 1. Simulación de Pérdida de Significancia
k_vals = np.arange(1, 21)
x_vals = 10.0**(-k_vals)
y_vals = ((1.0 + x_vals) - 1.0) / x_vals

fig, axs = plt.subplots(1, 2, figsize=(14, 6))

# Subplot 1: Colapso por machine epsilon
axs[0].semilogx(x_vals, y_vals, 'bo-', linewidth=1.5, markersize=6, label='Cálculo Computacional')
axs[0].axhline(1.0, color='red', linestyle='--', linewidth=1.5, label='Valor Analítico = 1.0')
axs[0].set_title('Pérdida de Significancia en $((1+x)-1)/x$', fontweight='bold', color='navy')
axs[0].set_xlabel('Valor de $x$ (Escala Logarítmica)')
axs[0].set_ylabel('Resultado obtenido')
axs[0].legend(loc='lower left')
axs[0].invert_xaxis()  # Para ver el límite hacia 0 de izquierda a derecha

# 2. Simulación de Cancelación en e^(-5.5)
import math
n_terms = np.arange(0, 40)
term_values_direct = []
sum_direct = 0.0
sums_direct_list = []

for n in n_terms:
    term = ((-5.5)**n) / float(math.factorial(n))
    term_values_direct.append(term)
    sum_direct += term
    sums_direct_list.append(sum_direct)

# Método recíproco
sum_recip = 0.0
sums_recip_list = []
for n in n_terms:
    term = (5.5**n) / float(math.factorial(n))
    sum_recip += term
    sums_recip_list.append(1.0 / sum_recip)

real_val = np.exp(-5.5)

axs[1].plot(n_terms, sums_direct_list, 'ro-', label='Suma Directa (Con oscilaciones y cancelación)')
axs[1].plot(n_terms, sums_recip_list, 'g^-', label='Suma Recíproca $1/e^{5.5}$ (Estable)')
axs[1].axhline(real_val, color='black', linestyle=':', label=f'Valor exacto $\\approx {real_val:.6f}$')
axs[1].set_title('Cálculo de $e^{-5.5}$ por Serie de Taylor', fontweight='bold', color='navy')
axs[1].set_xlabel('Número de términos ($N$)')
axs[1].set_ylabel('Suma acumulada $s_N$')
axs[1].set_ylim(-0.05, 0.1)
axs[1].legend()

plt.tight_layout()
plt.show()


## 3. Clasificación y Cuantificación de Errores

### 3.1 Clasificación General de Errores
1. **Errores del Modelo (Inherentes)**: Aquellos causados por las suposiciones y simplificaciones hechas al modelar matemáticamente un sistema físico real, o por imprecisiones en los datos de entrada obtenidos experimentalmente. Son imposibles de corregir numéricamente sin refinar el modelo o la medición.
2. **Errores del Método**: Originados por las limitaciones de realizar cálculos en máquinas digitales con precisión finita:
   - **Error de Truncamiento**: Provocado por aproximar un proceso matemático infinito (ej. sumas de series, derivadas por diferencias finitas) mediante un número limitado de pasos.
   - **Error de Redondeo**: Debido a la imposibilidad de representar números con infinitas cifras decimales en una memoria finita.

### 3.2 Regla de Redondeo Simétrico (Al Par más Cercano)
Para redondear un número a una cantidad determinada de cifras significativas, se analiza el primer dígito a descartar (dms):
1. Si el dms es **mayor que 5**, se incrementa en una unidad el dígito anterior.
2. Si el dms es **menor que 5**, el dígito anterior no se modifica.
3. Si el dms es **exactamente 5** (o 5 seguido únicamente de ceros):
   - Si el dígito anterior es **par**, se deja inalterado.
   - Si el dígito anterior es **impar**, se incrementa en una unidad (para hacerlo par).
   *Nota*: Este criterio evita sesgos sistemáticos de redondeo hacia arriba o hacia abajo en sumas repetidas.

### 3.3 Fórmulas de Cuantificación de Errores
Sea $V_t$ (o $P^*$) el valor real o tomado como exacto, y $V_a$ (o $P$) el valor aproximado:
- **Error Absoluto ($E_A$)**: Mide la magnitud de la desviación física y conserva las unidades de la medida:
  $$E_A = |V_t - V_a|$$
- **Error Relativo ($E_R$)**: Relaciona el error absoluto con el valor real, obteniendo una cantidad adimensional:
  $$E_R = \frac{|V_t - V_a|}{|V_t|} \quad (V_t \neq 0)$$
- **Error Porcentual ($E_P$)**: Expresa el error relativo en forma de porcentaje:
  $$E_P = E_R \times 100\%$$


## 4. Propagación de Errores y Orden de Operaciones

### 4.1 La Regla de Suma de McCracken
El orden en que se realizan las operaciones aritméticas en una computadora altera el resultado final. En la suma de múltiples cantidades de órdenes de magnitud muy dispares:
- **Suma Descendente**: Sumar de mayor a menor. Los números pequeños se suman al subtotal que ya es grande. Al alinearse los exponentes, las mantisas de los números pequeños se desplazan a la derecha perdiendo significancia (redondeados a cero).
- **Suma Ascendente**: Sumar de menor a mayor. Los números pequeños se acumulan entre sí primero, ganando peso numérico y modificando la mantisa del subtotal antes de sumarse a los números grandes. Esto **preserva la precisión y minimiza la propagación del error**.

### 4.2 Ejemplo Numérico del Texto
Sumar las siguientes cantidades utilizando aritmética de punto flotante normalizado de base 10 con mantisa de **4 dígitos significativos** y redondeo simétrico en cada paso intermedio:
1. $0.2685 \times 10^4 = 2685$
2. $0.9567 \times 10^3 = 956.7$
3. $0.0053 \times 10^2 = 0.53$
4. $0.1111 \times 10^1 = 1.111$

El **valor exacto** (precisión infinita) es:
$$S_{\text{exacto}} = 2685 + 956.7 + 0.53 + 1.111 = 3643.341$$

**Simulación paso a paso con redondeo a 4 dígitos**:

- **Suma Descendente** (Mayor a menor):
  1. Subtotal 1 = $0.2685 \times 10^4$
  2. Sumamos $0.09567 \times 10^4 \implies 0.36417 \times 10^4 \xrightarrow{\text{redondeo}} 0.3642 \times 10^4$ (3642)
  3. Sumamos $0.000053 \times 10^4 \implies 0.364253 \times 10^4 \xrightarrow{\text{redondeo al par}} 0.3643 \times 10^4$ (3643)
  4. Sumamos $0.0001111 \times 10^4 \implies 0.3644111 \times 10^4 \xrightarrow{\text{redondeo}} 0.3644 \times 10^4$ (3644)
  *Resultado Final Descendente*: **3644** ($E_A = 0.659$)

- **Suma Ascendente** (Menor a mayor):
  1. Subtotal 1 = $0.1111 \times 10^1$
  2. Sumamos $0.0530 \times 10^1 \implies 0.1641 \times 10^1$ (1.641)
  3. Sumamos $95.67 \times 10^1 \implies 95.8341 \times 10^1 \xrightarrow{\text{redondeo}} 95.83 \times 10^1$ (958.3)
  4. Sumamos $268.5 \times 10^1 \implies 364.33 \times 10^1 \xrightarrow{\text{redondeo}} 364.3 \times 10^1$ (3643)
  *Resultado Final Ascendente*: **3643** ($E_A = 0.341$)

El método ascendente arroja un error absoluto significativamente menor, validando la tesis de McCracken.

In [ ]:
# Implementación del Redondeo a 4 dígitos de mantisa y simulación de sumas
def round_to_sig(x, sig=4):
    if x == 0.0:
        return 0.0
    # Obtenemos el exponente de base 10
    exponent = int(np.floor(np.log10(abs(x))))
    mantissa = x / (10.0**exponent)
    # Redondeamos la mantisa a los dígitos de precisión deseados
    # Usamos round de numpy que aplica redondeo al par más cercano (round-to-even)
    rounded_mantissa = np.round(mantissa, sig - 1)
    return rounded_mantissa * (10.0**exponent)

numbers = [2685.0, 956.7, 0.53, 1.111]
exact_sum = sum(numbers)

# Suma Descendente (mayor a menor)
descending_nums = sorted(numbers, reverse=True)
subtotal_desc = descending_nums[0]
print("Suma Descendente (Paso a Paso):")
print(f"Inicio: {subtotal_desc:.4f}")
for val in descending_nums[1:]:
    raw_sum = subtotal_desc + val
    subtotal_desc = round_to_sig(raw_sum, 4)
    print(f"+ {val:<6} -> Subtotal: {subtotal_desc:.4f}")

err_desc = abs(exact_sum - subtotal_desc)

print("\nSuma Ascendente (Paso a Paso):")
ascending_nums = sorted(numbers)
subtotal_asc = ascending_nums[0]
print(f"Inicio: {subtotal_asc:.4f}")
for val in ascending_nums[1:]:
    raw_sum = subtotal_asc + val
    subtotal_asc = round_to_sig(raw_sum, 4)
    print(f"+ {val:<6} -> Subtotal: {subtotal_asc:.4f}")

err_asc = abs(exact_sum - subtotal_asc)

# Graficar comparación de errores
plt.figure(figsize=(8, 5.5))
plt.bar(['Suma Descendente', 'Suma Ascendente'], [err_desc, err_asc], color=['coral', 'mediumseagreen'], width=0.5)
plt.ylabel('Error Absoluto ($E_A$)')
plt.title('Comparación de Errores de Redondeo: McCracken', fontweight='bold')
for idx, val in enumerate([err_desc, err_asc]):
    plt.text(idx, val + 0.02, f'{val:.4f}', ha='center', fontweight='bold')
plt.ylim(0, 0.8)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


## 5. Convergencia, Estabilidad y Criterio de Scarborough

### 5.1 Definición de Convergencia y Estabilidad
- **Convergencia**: En un algoritmo iterativo, indica que las aproximaciones sucesivas se acercan progresivamente a la solución exacta. Matemáticamente, se cumple la condición de contracción del error:
  $$|x_n - x_{n-1}| \le |x_{n-1} - x_{n-2}|$$
- **Estabilidad**: Capacidad de un algoritmo para no amplificar los errores de redondeo introducidos en los pasos intermedios. Un sistema es estable si pequeñas variaciones en los datos iniciales provocan variaciones controladas y pequeñas en el resultado.

### 5.2 El Criterio de Tolerancia de Scarborough
En la práctica de la ingeniería, a menudo no se conoce el valor exacto para calcular el error. Por tanto, se mide el **error relativo porcentual aproximado ($e_a$)** entre dos iteraciones consecutivas:
$$e_a = \left| \frac{x_n - x_{n-1}}{x_n} \right| \times 100\%$$

**Criterio de Scarborough**: Para garantizar con total certeza matemática que un resultado aproximado es correcto en al menos $n$ cifras significativas, la iteración debe detenerse cuando el error aproximado $e_a$ sea menor que la tolerancia porcentual especificada:
$$\text{tol} = 0.5 \times 10^{2-n} \%$$

**Ejemplo**: Si deseamos un resultado con 5 cifras significativas exactas:
$$\text{tol} = 0.5 \times 10^{2-5}\% = 0.5 \times 10^{-3}\% = 0.0005\%$$


## 6. Polinomio de Taylor y Residuo

### 6.1 Definición
Si una función $f(x)$ posee derivadas continuas hasta el orden $n+1$ en un intervalo cerrado que contiene a los puntos $a$ y $x$, entonces la función puede representarse mediante su polinomio de Taylor de grado $n$ centrado en $x = a$ más un residuo o término de error $E_n(x)$:
$$f(x) = \sum_{k=0}^n \frac{f^{(k)}(a)}{k!} (x-a)^k + E_n(x)$$
donde el polinomio aproximador es:
$$P_n(x) = f(a) + f'(a)(x-a) + \frac{f''(a)}{2!}(x-a)^2 + \dots + \frac{f^{(n)}(a)}{n!}(x-a)^n$$

### 6.2 Demostración de la Forma Integral del Residuo
Queremos demostrar por inducción y mediante **integración por partes** que el residuo de orden $n$ viene dado por la expresión integral:
$$E_n(x) = \frac{1}{n!} \int_a^x (x-t)^n f^{(n+1)}(t) dt$$

**Prueba para el caso $n=1$**:
Por el Teorema Fundamental del Cálculo:
$$f(x) = f(a) + \int_a^x f'(t) dt$$
Reescribimos la integral mediante integración por partes definiendo:
- $u = f'(t) \implies du = f''(t) dt$
- $dv = dt \implies v = -(x - t)$  *(eligiendo la constante de integración convenientemente)*

Aplicando la fórmula $\int u dv = u v - \int v du$:
$$\int_a^x f'(t) dt = \left[ -f'(t)(x - t) \right]_a^x - \int_a^x -(x - t) f''(t) dt$$
$$\int_a^x f'(t) dt = 0 - \left( -f'(a)(x - a) \right) + \int_a^x (x - t) f''(t) dt$$
$$\int_a^x f'(t) dt = f'(a)(x-a) + \int_a^x (x-t) f''(t) dt$$

Sustituyendo esto en la expresión inicial para $f(x)$:
$$f(x) = f(a) + f'(a)(x-a) + \int_a^x (x-t) f''(t) dt$$
Identificando los términos, el residuo es exactamente:
$$E_1(x) = \int_a^x (x-t) f''(t) dt = \frac{1}{1!} \int_a^x (x-t)^1 f''(t) dt$$
Por inducción matemática, al aplicar integración por partes de forma sucesiva, se obtiene la expresión general para el orden $n$. Queda demostrado.

### 6.3 Cota de Error de Lagrange
Aplicando el teorema del valor medio para integrales a la forma integral del residuo, se obtiene la **forma de Lagrange**:
$$E_n(x) = \frac{f^{(n+1)}(\xi)}{(n+1)!} (x-a)^{n+1}$$
para algún valor de $\xi$ que pertenece al intervalo abierto entre $a$ y $x$. 

Si podemos acotar la magnitud de la derivada $(n+1)$-ésima por una constante $M$, es decir, $|f^{(n+1)}(t)| \le M$ para todo $t$ en el intervalo, entonces el error de truncamiento del polinomio está acotado superiormente por:
$$|E_n(x)| \le M \frac{|x-a|^{n+1}}{(n+1)!}$$

**Ejemplo**: Estimar el error máximo cometido al aproximar $\sin(x)$ con su polinomio de Taylor de grado 6 centrado en $a = 0$ (Maclaurin), para los puntos $x = \pi, \pi/2, \pi/4$.
- El desarrollo de Maclaurin es: $P_6(x) = x - \frac{x^3}{3!} + \frac{x^5}{5!}$ (puesto que el término de grado 6 es cero).
- La derivada de orden 7 de $\sin(x)$ es $-\cos(x)$, la cual está acotada en valor absoluto por $M = 1$.
- La cota de Lagrange del error $E_6(x)$ es:
  $$|E_6(x)| \le \frac{|x|^7}{7!} = \frac{|x|^7}{5040}$$
- Evaluamos en los puntos:
  1. Para $x = \pi$: $|E_6(\pi)| \le \frac{\pi^7}{5040} \approx 0.5993$
  2. Para $x = \pi/2$: $|E_6(\pi/2)| \le \frac{(\pi/2)^7}{5040} \approx 0.0047$
  3. Para $x = \pi/4$: $|E_6(\pi/4)| \le \frac{(\pi/4)^7}{5040} \approx 0.000037$
- Conclusión: A medida que el punto $x$ se acerca al centro $a = 0$, el error disminuye drásticamente, confirmando la convergencia local del polinomio.


In [ ]:
# 1. Gráfica de cos(x) y polinomios de Taylor en a = pi/2
x_plot = np.linspace(0, np.pi, 200)
y_cos = np.cos(x_plot)

# Taylor centrado en a = pi/2
# p1 = -(x - pi/2)
# p3 = -(x - pi/2) + (x - pi/2)^3 / 6
a = np.pi / 2.0
p1 = -(x_plot - a)
p3 = -(x_plot - a) + ((x_plot - a)**3) / 6.0

fig, axs = plt.subplots(1, 2, figsize=(14, 6.5))

axs[0].plot(x_plot, y_cos, 'k-', linewidth=2.5, label='$\\cos(x)$')
axs[0].plot(x_plot, p1, 'r--', linewidth=1.5, label='Polinomio Grado 1')
axs[0].plot(x_plot, p3, 'b-.', linewidth=1.5, label='Polinomio Grado 3')
axs[0].axvline(a, color='gray', linestyle=':', label='Centro $a = \\pi/2$')
axs[0].fill_between(x_plot, y_cos, p3, color='blue', alpha=0.1, label='Residuo $E_3(x)$')
axs[0].set_title('Aproximaciones de Taylor para $\\cos(x)$ en $a = \\pi/2$', fontweight='bold', color='navy')
axs[0].set_xlabel('$x$')
axs[0].set_ylabel('$y$')
axs[0].set_ylim(-1.5, 1.5)
axs[0].legend(loc='lower left')

# 2. Cota de error de Lagrange para sin(x) de grado 6 en a = 0
x_lag = np.linspace(-np.pi, np.pi, 300)
y_sin = np.sin(x_lag)
p6_sin = x_lag - (x_lag**3) / 6.0 + (x_lag**5) / 120.0
error_actual = np.abs(y_sin - p6_sin)
cota_lagrange = (np.abs(x_lag)**7) / 5040.0

axs[1].plot(x_lag, error_actual, 'g-', linewidth=2.0, label='Error Real $|\\sin(x) - P_6(x)|$')
axs[1].plot(x_lag, cota_lagrange, 'r--', linewidth=1.5, label='Cota de Lagrange $x^7 / 7!$')

# Puntos específicos de control
control_points = [np.pi/4, np.pi/2, np.pi]
colores = ['darkorange', 'magenta', 'red']
for pt, col in zip(control_points, colores):
    err_pt = np.abs(np.sin(pt) - (pt - (pt**3)/6.0 + (pt**5)/120.0))
    cota_pt = (pt**7) / 5040.0
    axs[1].scatter(pt, err_pt, color=col, s=50, zorder=5)
    axs[1].text(pt + 0.15, err_pt, f'x={pt/np.pi:.2f}\\pi\\nErr: {err_pt:.5f}', fontsize=9, fontweight='bold', color=col)

axs[1].set_title('Error Real vs. Cota de Lagrange para $\\sin(x)$ ($N=6$)', fontweight='bold', color='navy')
axs[1].set_xlabel('$x$')
axs[1].set_ylabel('Magnitud del error')
axs[1].set_yscale('log')
axs[1].set_ylim(1e-8, 10)
axs[1].legend(loc='upper left')

plt.tight_layout()
plt.show()


## 7. Resumen de la Lección

1. **Análisis Numérico**: Disciplina enfocada en algoritmos iterativos para problemas continuos que entregan aproximaciones de alta precisión, esenciales para ecuaciones complejas y trascendentes.
2. **Aritmética Limitada**: La precisión del punto flotante computacional está acotada por el épsilon de máquina, provocando colapso de significancia y errores de cancelación restativa al restar números similares o sumar escalas dispares.
3. **Errores**: Clasificados en inherentes (modelo) y del método (truncamiento y redondeo). El redondeo simétrico al par más cercano elimina sesgos sistemáticos.
4. **Regra de McCracken**: El orden de operaciones altera los resultados. Sumar en orden ascendente (menor a mayor) acumula valores pequeños y reduce drásticamente el error de redondeo acumulado.
5. **Criterio de Scarborough**: Permite garantizar con total seguridad analítica que un resultado es exacto hasta $n$ cifras significativas deteniendo la iteración cuando $e_a < 0.5 \times 10^{2-n} \%$.
6. **Residuo de Taylor**: Representa el error de truncamiento del polinomio. Puede formularse rigurosamente de manera integral e integrando por partes, o acotarse de forma sencilla mediante la cota de Lagrange.

## 8. Referencias Bibliográficas

1. Castillo, D. et al. (2010). *Introducción al Análisis Numérico*. Facultad de Educación y Humanidades, Universidad del Bío Bío. [En línea]. Disponible en: http://repobib.ubiobio.cl/jspui/bitstream/123456789/1955/3/Rivas_Urra_Darwin.pdf
2. Cortés, G. et al. (2019). *Aproximación numérica y errores*. División de Ciencias Básicas, Facultad de Ingeniería, UNAM. [En línea]. Disponible en: https://www.ingenieria.unam.mx/pinilla/PE105117/pdfs/tema1/1_aproximacion_numerica_y_errores.pdf
3. Zuazua, E. (2004). *Una introducción histórica al Análisis Numérico, el Control y su docencia*. Dpto. de Matemáticas, Universidad Autónoma de Madrid. [En línea]. Disponible en: https://verso.mat.uam.es/web/ezuazua/documentos_public/archivos/personal/comites/refelxiones.pdf
4. De la Fuente, J. (2017). *Ingeniería de los algoritmos y métodos numéricos*. Segunda edición, Editorial Círculo Rojo. [En línea]. Disponible en: http://www.jldelafuenteoconnor.es/Libro2017_NV_10-8_SP.pdf
5. Velásquez, A. (2008). *Análisis numérico: notas de clase*. Universidad del Norte. [En línea]. Disponible en: https://docplayer.es/21089629-An-alisis-numerico-notas-de-clase.html
6. Suárez, F. et al. (2012). *Tipos de errores: error absoluto, relativo, porcentual, redondeo y truncamiento*. Dpto. de Ciencias Básicas, Tecnológico de Tuxtla Gutiérrez. [En línea]. Disponible en: https://sites.google.com/site/metalnumericos/home/unidad-1/1-2-tipos-de-errores-error-absoluto-error-relativo-error-porcentual-errores-de-redondeo-y-truncamiento
